# Question 4: Conversational RAG System with Chat History and Follow-Up Questions

In [1]:
# Create ten documents

from pathlib import Path

DOCS_DIR = Path("conversational_rag_docs")
DOCS_DIR.mkdir(exist_ok=True)

documents_data = {
    "01_machine_learning.txt": """
Machine Learning
Machine learning is a branch of artificial intelligence that enables
systems to learn patterns from data and make predictions or decisions.
Common applications include fraud detection, recommendation systems,
forecasting, and image recognition.
""",
    "02_ml_types.txt": """
Three Main Types of Machine Learning
The three major types are supervised learning, unsupervised learning,
and reinforcement learning. Supervised learning uses labeled examples.
Unsupervised learning discovers patterns in unlabeled data.
Reinforcement learning trains an agent using rewards and penalties.
""",
    "03_supervised_learning.txt": """
Supervised Learning
Supervised learning uses input data paired with known target labels.
Classification predicts categories, while regression predicts numeric
values. Examples include spam detection and house-price prediction.
""",
    "04_unsupervised_learning.txt": """
Unsupervised Learning
Unsupervised learning identifies patterns in data without known target
labels. Clustering groups similar observations. For example, a retailer
could group customers based on purchasing behavior without predefined
customer categories.
""",
    "05_reinforcement_learning.txt": """
Reinforcement Learning
Reinforcement learning trains an agent to choose actions in an
environment. The agent receives rewards or penalties and learns a
policy to maximize cumulative reward. Applications include robotics
and game-playing systems.
""",
    "06_ml_advantages.txt": """
Advantages of Machine Learning
Machine learning can automate repetitive prediction tasks, identify
complex patterns, personalize recommendations, and improve decisions
when trained on suitable data. Performance depends on data quality,
evaluation, and monitoring. Models can also make mistakes or reproduce
biases in training data.
""",
    "07_ml_limitations.txt": """
Limitations of Machine Learning
Machine learning systems may require substantial data and computation.
They can overfit training data, perform poorly on changing data, and
produce biased predictions. Testing, monitoring, privacy protections,
and human oversight are important in appropriate applications.
""",
    "08_model_evaluation.txt": """
Model Evaluation
A model should be evaluated on data not used to fit its parameters.
Classification metrics include accuracy, precision, recall, and F1.
Regression metrics include mean absolute error and root mean squared
error. The appropriate metric depends on the task and error costs.
""",
    "09_data_preprocessing.txt": """
Data Preprocessing
Data preprocessing includes cleaning records, handling missing values,
encoding categorical variables, and scaling numerical features.
Preprocessing decisions should be learned from training data and applied
consistently to avoid data leakage.
""",
    "10_ai_applications.txt": """
Artificial Intelligence Applications
AI and machine learning are used in healthcare support, agriculture,
transportation, education, finance, and manufacturing. Systems should
be evaluated for reliability, privacy, fairness, and suitability for
their intended use.
"""
}

for filename, content in documents_data.items():
    (DOCS_DIR / filename).write_text(
        content.strip(),
        encoding="utf-8"
    )

print("Documents created:", len(documents_data))
print("Folder:", DOCS_DIR.resolve())


Documents created: 10
Folder: C:\Users\HP\conversational_rag_docs


In [2]:
# Load, split, embed, and store documents

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

loaded_documents = []

for path in sorted(DOCS_DIR.glob("*.txt")):
    loaded_documents.append(
        Document(
            page_content=path.read_text(encoding="utf-8"),
            metadata={"source": path.name}
        )
    )

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60
)

chunks = splitter.split_documents(loaded_documents)

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="conversational_rag",
    persist_directory="conversational_rag_db"
)

print("Documents loaded:", len(loaded_documents))
print("Chunks stored:", len(chunks))


Documents loaded: 10
Chunks stored: 10


In [3]:
# Implement conversation history and query rewriting

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

chat_history = []

def rewrite_question(question, history):
    if not history:
        return question

    history_text = "\n".join(
        f"User: {item['question']}\nAssistant: {item['answer']}"
        for item in history[-5:]
    )

    prompt = f"""
Rewrite the latest question as a standalone search query.
Use conversation history to resolve references such as "it",
"its", "the second type", or "that model".
Keep the original meaning. Return only the rewritten query.

Conversation history:
{history_text}

Latest question:
{question}

Standalone search query:
"""

    return llm.invoke(prompt).content.strip()

print("Conversation memory and query rewriting are ready.")


Conversation memory and query rewriting are ready.


In [4]:
# Retrieve evidence and generate grounded answers

def conversational_rag(question):
    rewritten_query = rewrite_question(question, chat_history)

    retrieved_docs = vectorstore.similarity_search(
        rewritten_query,
        k=3
    )

    if not retrieved_docs:
        answer = (
            "Insufficient evidence: no relevant documents were retrieved."
        )
        chat_history.append({
            "question": question,
            "answer": answer
        })
        return {
            "original_question": question,
            "rewritten_query": rewritten_query,
            "answer": answer,
            "sources": []
        }

    context = "\n\n".join(
        f"[S{i}] Source: {doc.metadata['source']}\n"
        f"{doc.page_content}"
        for i, doc in enumerate(retrieved_docs, start=1)
    )

    history_text = "\n".join(
        f"User: {item['question']}\nAssistant: {item['answer']}"
        for item in chat_history[-5:]
    )

    prompt = f"""
You are a document-grounded conversational assistant.

Use retrieved evidence as the factual basis for your answer.
Use conversation history to understand references and context.
Do not treat previous assistant answers as independent evidence.
Cite document evidence with labels such as [S1].
If the documents do not support an answer, explicitly state
that there is insufficient evidence.

Conversation history:
{history_text}

Current question:
{question}

Rewritten search query:
{rewritten_query}

Retrieved evidence:
{context}

Answer:
"""

    answer = llm.invoke(prompt).content

    chat_history.append({
        "question": question,
        "answer": answer
    })

    return {
        "original_question": question,
        "rewritten_query": rewritten_query,
        "answer": answer,
        "sources": [
            {
                "source": doc.metadata["source"],
                "excerpt": doc.page_content
            }
            for doc in retrieved_docs
        ]
    }


In [5]:
# Test five consecutive questions

conversation_questions = [
    "What is machine learning?",
    "What are its three main types?",
    "Explain the second type with an example.",
    "What are its advantages?",
    "Which document supports the explanation?"
]

conversation_results = []

for number, question in enumerate(conversation_questions, start=1):
    result = conversational_rag(question)
    conversation_results.append(result)

    print(f"\n{'=' * 60}")
    print(f"QUESTION {number}: {question}")
    print("REWRITTEN QUERY:", result["rewritten_query"])
    print("ANSWER:", result["answer"])

    print("SOURCE EVIDENCE:")
    for source in result["sources"]:
        print("Source:", source["source"])
        print("Excerpt:", source["excerpt"])



QUESTION 1: What is machine learning?
REWRITTEN QUERY: What is machine learning?
ANSWER: Based on the retrieved evidence, machine learning is a branch of artificial intelligence that enables systems to learn patterns from data and make predictions or decisions. This is stated in [S1] as: "Machine Learning is a branch of artificial intelligence that enables systems to learn patterns from data and make predictions or decisions."

Additionally, [S2] highlights the advantages of machine learning, including automation of repetitive tasks, identification of complex patterns, and personalization of recommendations. However, [S3] notes limitations of machine learning, such as the need for substantial data and computation, potential overfitting, and biased predictions.

In summary, machine learning is a field that allows systems to learn from data and make predictions or decisions, with both benefits and limitations.
SOURCE EVIDENCE:
Source: 01_machine_learning.txt
Excerpt: Machine Learning
Ma

In [6]:
# Compare retrieval with and without rewriting

followup = "Explain the second type with an example."

# Direct retrieval uses the original follow-up question.
without_rewriting = vectorstore.similarity_search(
    followup,
    k=3
)

# Rewritten retrieval uses the conversation history.
standalone_query = rewrite_question(
    followup,
    chat_history[:2]
)

with_rewriting = vectorstore.similarity_search(
    standalone_query,
    k=3
)

print("Original query:", followup)
print("Rewritten query:", standalone_query)

print("\nWITHOUT QUERY REWRITING:")
for doc in without_rewriting:
    print(doc.metadata["source"], ":", doc.page_content[:200])

print("\nWITH QUERY REWRITING:")
for doc in with_rewriting:
    print(doc.metadata["source"], ":", doc.page_content[:200])


Original query: Explain the second type with an example.
Rewritten query: What is unsupervised learning with an example?

WITHOUT QUERY REWRITING:
02_ml_types.txt : Three Main Types of Machine Learning
The three major types are supervised learning, unsupervised learning,
and reinforcement learning. Supervised learning uses labeled examples.
Unsupervised learning 
03_supervised_learning.txt : Supervised Learning
Supervised learning uses input data paired with known target labels.
Classification predicts categories, while regression predicts numeric
values. Examples include spam detection a
09_data_preprocessing.txt : Data Preprocessing
Data preprocessing includes cleaning records, handling missing values,
encoding categorical variables, and scaling numerical features.
Preprocessing decisions should be learned from

WITH QUERY REWRITING:
04_unsupervised_learning.txt : Unsupervised Learning
Unsupervised learning identifies patterns in data without known target
labels. Clustering groups si

In [7]:
# Save the transcript and retrieval comparison

import json
import pandas as pd

transcript_df = pd.DataFrame([
    {
        "Turn": i,
        "Question": result["original_question"],
        "Rewritten_Query": result["rewritten_query"],
        "Answer": result["answer"],
        "Sources": "; ".join(
            source["source"] for source in result["sources"]
        )
    }
    for i, result in enumerate(conversation_results, start=1)
])

transcript_df.to_csv(
    "conversational_rag_transcript.csv",
    index=False
)

comparison_df = pd.DataFrame([
    {
        "Method": "Without rewriting",
        "Query": followup,
        "Retrieved_Sources": "; ".join(
            doc.metadata["source"] for doc in without_rewriting
        )
    },
    {
        "Method": "With rewriting",
        "Query": standalone_query,
        "Retrieved_Sources": "; ".join(
            doc.metadata["source"] for doc in with_rewriting
        )
    }
])

comparison_df.to_csv(
    "followup_retrieval_comparison.csv",
    index=False
)

print("Transcript saved.")
print("Retrieval comparison saved.")


Transcript saved.
Retrieval comparison saved.
